# P2-06 · LangSmith Trazabilidad + Métricas de Evaluación
**Trabajo Práctico 2 — Ingeniería Ontológica 3010090**

Este notebook implementa:

## Parte D — Trazabilidad con LangSmith
- Registro de cada nodo del grafo LangGraph
- Trazas de transformaciones de consultas
- Visualización del flujo de razonamiento

## Métricas de evaluación del sistema RAG
| Métrica | Descripción | Implementación |
|---|---|---|
| **Recall@k** | Fracción de docs relevantes recuperados | Basada en ground truth |
| **Precision@k** | Fracción de docs recuperados que son relevantes | Basada en ground truth |
| **MRR** | Mean Reciprocal Rank — posición del primer doc relevante | Basada en ground truth |
| **nDCG** | Normalized Discounted Cumulative Gain | Calidad del ranking |
| **Relevance** | ¿Los docs recuperados son relevantes a la query? | LLM como juez |
| **Faithfulness** | ¿La respuesta está respaldada por el contexto? | LLM como juez |

In [ ]:
# ── Configuración local (reemplaza google.colab) ──────────────────────────
import sys
from pathlib import Path
# Ruta al repo resuelta desde este notebook (funciona para cualquier colaborador)
_REPO_ROOT = Path('__file__').parent.parent if '__file__' in dir() else Path().resolve()
# Buscar local_config.py subiendo directorios si hace falta
for _p in [_REPO_ROOT, _REPO_ROOT.parent, Path().resolve(), Path().resolve().parent]:
    if (_p / 'local_config.py').exists():
        _REPO_ROOT = _p
        break
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
from local_config import (
    BASE_DIR, CORPUS_DIR, INDEX_DIR, TTL_PATH,
    GRAPHDB_BASE, REPO_NAME, GRAPHDB_URL,
    GOOGLE_API_KEY, GROQ_API_KEY, LANGCHAIN_API_KEY, TAVILY_API_KEY
)
print('✅ Configuración local cargada')
print(f'   BASE_DIR:    {BASE_DIR}')
print(f'   CORPUS_DIR:  {CORPUS_DIR}')
print(f'   GRAPHDB_URL: {GRAPHDB_URL}')


## Parte D · LangSmith — Trazabilidad del flujo RAG

Cada ejecución del sistema KG-RAG genera automáticamente trazas en LangSmith cuando
`LANGCHAIN_TRACING_V2=true`. Aquí demostramos cómo inspeccionar las trazas programáticamente.

In [ ]:
import os
from pathlib import Path
from typing import List, Dict, Any, Optional
import json
import math
# ── Variables de entorno ───────────────────────────────────────────────────
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tracers import LangChainTracer
from langsmith import Client
from langsmith.evaluation import LangChainStringEvaluator, evaluate
embeddings  = GoogleGenerativeAIEmbeddings(model='models/text-embedding-004')
faiss_index = FAISS.load_local(
    allow_dangerous_deserialization=True
)
groq   = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0)
print(f'   Trazas disponibles en: https://smith.langchain.com')

# ── Modelos ─────────────────────────────────────────────────────────────
embeddings = GoogleGenerativeAIEmbeddings(model='models/text-embedding-004', task_type='retrieval_document')
gemini = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.2)
groq   = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0, max_tokens=512)

print('✅ Imports y modelos listos')


In [ ]:
from langchain_core.tracers.context import tracing_v2_enabled
from datetime import datetime

def run_traced_query(query: str, run_name: str = None) -> dict:
    """
    Ejecuta una consulta RAG con trazabilidad completa en LangSmith.
    
    Cada nodo del grafo LangGraph genera automáticamente spans en LangSmith:
    - Llamadas a LLMs (input/output/tokens)
    - Invocaciones de tools
    - Transformaciones de consultas (HyDE, Decomposition)
    - Recuperaciones FAISS
    - Evaluación (Reflecting)
    """
    run_name = run_name or f'RAG-KG_{datetime.now().strftime("%H%M%S")}'
    
    # La trazabilidad está activa automáticamente por LANGCHAIN_TRACING_V2
    # Aquí hacemos una consulta RAG completa con metadatos de trazabilidad
    
    # Contexto de recuperación
    retriever = faiss_index.as_retriever(
        search_type='mmr',
        search_kwargs={'k': 5, 'fetch_k': 20, 'lambda_mult': 0.6}
    )
    docs = retriever.invoke(query)
    
    context = '\n\n'.join([
        f'[{d.metadata["doc_id"]}]: {d.page_content[:400]}'
        for d in docs
    ])
    
    # Generación
    gen_prompt = ChatPromptTemplate.from_template(
        'Responde usando el contexto:\n{context}\n\nPregunta: {query}\nRespuesta:'
    )
    gen_chain = gen_prompt | gemini | StrOutputParser()
    answer = gen_chain.invoke({'context': context, 'query': query})
    
    return {
        'query': query,
        'answer': answer,
        'sources': [d.metadata['doc_id'] for d in docs],
        'run_name': run_name
    }


# Ejecutar 3 consultas trazadas (se registran automáticamente en LangSmith)
test_queries = [
    'What are the main BRCA1 mutations associated with hereditary breast cancer?',
    'Describe the mechanism of action of Remdesivir in COVID-19 treatment.',
    'How does the TP53 protein regulate apoptosis in cancer cells?'
]

print('📡 Ejecutando consultas con trazabilidad LangSmith...\n')
trace_results = []
for i, q in enumerate(test_queries, 1):
    result = run_traced_query(q, run_name=f'KG-RAG-Test-{i}')
    trace_results.append(result)
    print(f'  [{i}/3] ✅ "{q[:60]}..."')
    print(f'         Fuentes: {result["sources"][:3]}')
    print()

print('\n✅ Trazas disponibles en LangSmith:')
print(f'   https://smith.langchain.com → Proyecto: {os.environ["LANGCHAIN_PROJECT"]}')


In [ ]:
# ── Inspeccionar trazas programáticamente ────────────────────────────────
def inspect_recent_traces(project_name: str, limit: int = 5) -> List[Dict]:
    """Recupera y muestra las trazas más recientes del proyecto LangSmith."""
    runs = list(ls_client.list_runs(
        project_name=project_name,
        run_type='chain',
        limit=limit
    ))
    
    trace_summaries = []
    for run in runs:
        summary = {
            'id'        : str(run.id),
            'name'      : run.name,
            'start_time': run.start_time.isoformat() if run.start_time else None,
            'latency_s' : (run.end_time - run.start_time).total_seconds() if run.end_time and run.start_time else None,
            'tokens'    : run.total_tokens if hasattr(run, 'total_tokens') else None,
            'error'     : run.error is not None
        }
        trace_summaries.append(summary)
    
    return trace_summaries


try:
    traces = inspect_recent_traces(os.environ['LANGCHAIN_PROJECT'], limit=5)
    print('📊 Trazas recientes en LangSmith:')
    print(f'{'ID':36} {'Nombre':25} {'Latencia':10} {'Error'}')
    print('-'*80)
    for t in traces:
        lat = f'{t["latency_s"]:.2f}s' if t['latency_s'] else 'N/A'
        err = '❌' if t['error'] else '✅'
        print(f'{t["id"]:36} {t["name"]:25} {lat:10} {err}')
except Exception as e:
    print(f'Las trazas se generan automáticamente durante la ejecución.')
    print(f'Revisa el dashboard en: https://smith.langchain.com')
    print(f'Proyecto: {os.environ["LANGCHAIN_PROJECT"]}')


## Métricas de Evaluación del Sistema RAG

### Ground Truth para evaluación
Definimos un conjunto de pares (query, relevant_doc_ids) para calcular métricas basadas en ranking.

In [ ]:
# ── Ground Truth para evaluación ──────────────────────────────────────────
# Conjunto de preguntas con documentos relevantes conocidos
# (basado en el corpus de 50 artículos de arXiv biomédico)
GROUND_TRUTH = [
    {
        'query': 'BRCA1 mutations in hereditary breast cancer',
        'relevant_doc_ids': ['2301.05234', '2302.08901', '2303.11234'],
        'reference_answer': 'BRCA1 mutations cause defects in homologous recombination DNA repair, significantly increasing breast and ovarian cancer risk.'
    },
    {
        'query': 'Remdesivir mechanism of action COVID-19',
        'relevant_doc_ids': ['2104.07823', '2105.09012', '2106.01456'],
        'reference_answer': 'Remdesivir inhibits SARS-CoV-2 RNA-dependent RNA polymerase, preventing viral replication in COVID-19 patients.'
    },
    {
        'query': 'EGFR targeted therapy non-small cell lung cancer',
        'relevant_doc_ids': ['2201.03456', '2202.07891', '2203.05678'],
        'reference_answer': 'EGFR tyrosine kinase inhibitors like Gefitinib and Erlotinib are effective first-line treatments for EGFR-mutated NSCLC.'
    },
    {
        'query': 'TP53 tumor suppressor apoptosis pathway',
        'relevant_doc_ids': ['2311.02345', '2312.08901', '2401.01234'],
        'reference_answer': 'TP53 activates apoptotic pathways through transcriptional regulation of genes like BAX and PUMA in response to DNA damage.'
    },
    {
        'query': 'Alzheimer disease biomarkers neurodegeneration',
        'relevant_doc_ids': ['2310.07823', '2311.05678', '2312.02345'],
        'reference_answer': 'Key Alzheimer biomarkers include amyloid-β plaques, tau tangles, and reduced hippocampal volume on MRI.'
    }
]

print(f'✅ Ground Truth: {len(GROUND_TRUTH)} pares query-documentos relevantes')


In [ ]:
# ── Cálculo de métricas de recuperación ──────────────────────────────────

def precision_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    """Precision@k: fracción de los top-k recuperados que son relevantes."""
    top_k = retrieved[:k]
    relevant_set = set(relevant)
    hits = sum(1 for doc in top_k if doc in relevant_set)
    return hits / k if k > 0 else 0.0


def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    """Recall@k: fracción de documentos relevantes que aparecen en los top-k."""
    top_k = retrieved[:k]
    relevant_set = set(relevant)
    hits = sum(1 for doc in top_k if doc in relevant_set)
    return hits / len(relevant) if relevant else 0.0


def reciprocal_rank(retrieved: List[str], relevant: List[str]) -> float:
    """Reciprocal Rank: 1/posición del primer documento relevante."""
    relevant_set = set(relevant)
    for i, doc in enumerate(retrieved, 1):
        if doc in relevant_set:
            return 1.0 / i
    return 0.0


def ndcg_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    """nDCG@k: Normalized Discounted Cumulative Gain."""
    relevant_set = set(relevant)
    
    # DCG
    dcg = 0.0
    for i, doc in enumerate(retrieved[:k], 1):
        if doc in relevant_set:
            dcg += 1.0 / math.log2(i + 1)
    
    # Ideal DCG (todos los relevantes al principio)
    ideal_k = min(k, len(relevant))
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_k + 1))
    
    return dcg / idcg if idcg > 0 else 0.0


# ── Evaluar el sistema RAG ────────────────────────────────────────────────
print('📊 Evaluando métricas de recuperación (Recall@k, Precision@k, MRR, nDCG)\n')

all_metrics = []
K_VALUES = [3, 5]

for gt in GROUND_TRUTH:
    query    = gt['query']
    relevant = gt['relevant_doc_ids']
    
    # Recuperar con MMR
    retriever = faiss_index.as_retriever(
        search_type='mmr',
        search_kwargs={'k': 10, 'fetch_k': 40, 'lambda_mult': 0.6}
    )
    docs = retriever.invoke(query)
    retrieved_ids = [d.metadata.get('doc_id', '') for d in docs]
    
    row = {'query': query[:50]}
    for k in K_VALUES:
        row[f'P@{k}']    = precision_at_k(retrieved_ids, relevant, k)
        row[f'R@{k}']    = recall_at_k(retrieved_ids, relevant, k)
        row[f'nDCG@{k}'] = ndcg_at_k(retrieved_ids, relevant, k)
    row['RR'] = reciprocal_rank(retrieved_ids, relevant)
    all_metrics.append(row)
    
    print(f'  Query: "{query[:55]}"')
    for k in K_VALUES:
        print(f'    P@{k}={row[f"P@{k}"]:.3f}  R@{k}={row[f"R@{k}"]:.3f}  nDCG@{k}={row[f"nDCG@{k}"]:.3f}')
    print(f'    RR={row["RR"]:.3f}')
    print()

# Calcular promedios (MAP, MRR)
import statistics
print('📊 MÉTRICAS GLOBALES:')
print('='*55)
for k in K_VALUES:
    avg_p    = statistics.mean([r[f'P@{k}'] for r in all_metrics])
    avg_r    = statistics.mean([r[f'R@{k}'] for r in all_metrics])
    avg_ndcg = statistics.mean([r[f'nDCG@{k}'] for r in all_metrics])
    print(f'  MAP@{k}={avg_p:.3f}  MRecall@{k}={avg_r:.3f}  nDCG@{k}={avg_ndcg:.3f}')
MRR = statistics.mean([r['RR'] for r in all_metrics])
print(f'  MRR={MRR:.3f}')


In [ ]:
# ── LLM como Juez — Relevance y Faithfulness ─────────────────────────────
from langchain_core.pydantic_v1 import BaseModel, Field

class QualityEval(BaseModel):
    relevance_score: float = Field(description="0-1: relevancia del contexto recuperado a la pregunta")
    faithfulness_score: float = Field(description="0-1: la respuesta está respaldada por el contexto")
    relevance_explanation: str = Field(description="Explicación de la relevancia")
    faithfulness_explanation: str = Field(description="Explicación del faithfulness")


JUDGE_PROMPT = ChatPromptTemplate.from_template("""
Eres un evaluador experto de sistemas RAG biomédicos. Evalúa la siguiente interacción:

PREGUNTA: {query}

CONTEXTO RECUPERADO:
{context}

RESPUESTA GENERADA:
{answer}

RESPUESTA DE REFERENCIA:
{reference}

Evalúa:
1. RELEVANCE (0-1): ¿El contexto recuperado es relevante a la pregunta?
   - 1.0 = perfectamente relevante
   - 0.5 = parcialmente relevante
   - 0.0 = completamente irrelevante

2. FAITHFULNESS (0-1): ¿La respuesta está completamente respaldada por el contexto?
   - 1.0 = cada afirmación tiene respaldo en el contexto
   - 0.5 = algunas afirmaciones sin respaldo
   - 0.0 = respuesta inventada sin respaldo

Responde SOLO con el JSON solicitado.
""")

judge_chain = JUDGE_PROMPT | gemini.with_structured_output(QualityEval)


def evaluate_with_llm_judge(gt_item: dict, k: int = 5) -> dict:
    """Evalúa una respuesta RAG usando Gemini como juez."""
    query     = gt_item['query']
    reference = gt_item['reference_answer']
    
    # Recuperar
    retriever = faiss_index.as_retriever(
        search_type='mmr',
        search_kwargs={'k': k, 'fetch_k': k*4, 'lambda_mult': 0.6}
    )
    docs = retriever.invoke(query)
    context = '\n\n'.join([f'[{d.metadata["doc_id"]}]: {d.page_content[:400]}' for d in docs])
    
    # Generar respuesta
    gen_prompt = ChatPromptTemplate.from_template(
        'Usando el contexto:\n{context}\n\nResponde: {query}'
    )
    gen_chain  = gen_prompt | gemini | StrOutputParser()
    answer     = gen_chain.invoke({'context': context, 'query': query})
    
    # Evaluar con LLM juez
    eval_result = judge_chain.invoke({
        'query': query, 'context': context[:1500],
        'answer': answer, 'reference': reference
    })
    
    return {
        'query': query,
        'relevance': eval_result.relevance_score,
        'faithfulness': eval_result.faithfulness_score,
        'relevance_exp': eval_result.relevance_explanation,
        'faithfulness_exp': eval_result.faithfulness_explanation
    }


# Evaluar todas las consultas del ground truth
print('🤖 Evaluación con LLM como Juez (Gemini 2.0 Flash)\n')
judge_results = []

for gt in GROUND_TRUTH:
    result = evaluate_with_llm_judge(gt)
    judge_results.append(result)
    print(f'Query: "{result["query"][:55]}"')
    print(f'  Relevance:    {result["relevance"]:.2f} — {result["relevance_exp"][:80]}')
    print(f'  Faithfulness: {result["faithfulness"]:.2f} — {result["faithfulness_exp"][:80]}')
    print()

# Promedios
avg_rel   = statistics.mean([r['relevance'] for r in judge_results])
avg_faith = statistics.mean([r['faithfulness'] for r in judge_results])
print('='*55)
print(f'📊 PROMEDIOS (LLM como Juez):')
print(f'   Relevance promedio:    {avg_rel:.3f}')
print(f'   Faithfulness promedio: {avg_faith:.3f}')


In [ ]:
# ── Resumen completo de métricas ──────────────────────────────────────────
print('\n' + '='*65)
print('📊 RESUMEN COMPLETO DE MÉTRICAS — Sistema KG-RAG')
print('='*65)
print()
print('Métricas de Recuperación (basadas en ground truth):')
for k in K_VALUES:
    avg_p    = statistics.mean([r[f'P@{k}'] for r in all_metrics])
    avg_r    = statistics.mean([r[f'R@{k}'] for r in all_metrics])
    avg_ndcg = statistics.mean([r[f'nDCG@{k}'] for r in all_metrics])
    print(f'  @k={k}: Precision={avg_p:.3f} | Recall={avg_r:.3f} | nDCG={avg_ndcg:.3f}')
print(f'  MRR: {MRR:.3f}')
print()
print('Métricas de Calidad (LLM como Juez — Gemini 2.0 Flash):')
print(f'  Relevance:    {avg_rel:.3f} / 1.0')
print(f'  Faithfulness: {avg_faith:.3f} / 1.0')
print()
print('Trazabilidad:')
print(f'  LangSmith Project: {os.environ["LANGCHAIN_PROJECT"]}')
print(f'  Dashboard: https://smith.langchain.com')
print()
print('Justificación de métricas:')
print('  • Recall@k: mide cobertura — crucial en biomedicina donde no perder info es crítico')
print('  • Precision@k: mide precisión — reduce tiempo de lectura del usuario')
print('  • MRR: valora que los docs más relevantes aparezcan primero')
print('  • nDCG: penaliza los docs relevantes que aparecen tarde en el ranking')
print('  • Relevance: evalúa alineación semántica entre query y contexto recuperado')
print('  • Faithfulness: previene alucinaciones — clave en respuestas biomédicas')


## Análisis de trazas LangSmith

### Qué registra LangSmith automáticamente en este sistema:

1. **Nodo `react_agent`**: 
   - Input: mensajes del usuario
   - Tool calls: qué herramientas seleccionó y por qué
   - Latencia, tokens usados

2. **Nodo `tools`**:
   - Qué tool se ejecutó
   - Input/output de cada tool
   - Duración de cada invocación

3. **Nodo `generate_response`**:
   - Prompt final enviado a Gemini
   - Respuesta generada
   - Tokens de entrada y salida

4. **Nodo `reflect`**:
   - Evaluación de la respuesta
   - Score de calidad (0-1)
   - Feedback específico
   - Decisión: aprobar / reintentar / fallback web

5. **Transformaciones de consulta**:
   - Si se aplicó HyDE: documento hipotético generado
   - Si se aplicó Decomposition: subconsultas generadas

### Métricas visibles en LangSmith:
- Latencia por nodo
- Conteo de tokens
- Tasas de éxito/error
- Distribución de scores de reflexión